# Figure 4: Benchmark metrics

Run after the training and evaluation commands in `bash/paper/`. SCENE outputs use the `scLDM` names referenced below.


In [ ]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "environment_scene.yaml").exists())
os.chdir(PROJECT_ROOT / "notebooks" / "figures")
for folder in ("fig_1", "fig_2", "fig_3", "fig_4", "fig_5", "fig_6", "app", "qc"):
    Path("output", folder).mkdir(parents=True, exist_ok=True)


## Aggregate benchmark metrics


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


PROJECT_ROOT = PROJECT_ROOT
OUT_DIR = PROJECT_ROOT / "notebooks" / "figures" / "output" / "fig_4"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def get_error(x: pd.Series, kind: str = "sem") -> float:
    x = x.dropna()

    if len(x) <= 1:
        return np.nan

    if kind == "std":
        return x.std(ddof=1)
    if kind == "sem":
        return x.sem(ddof=1)
    if kind == "ci95":
        return 1.96 * x.sem(ddof=1)

    raise ValueError(f"Unknown error type: {kind}")


def summarize_benchmark_dataset(
    dataset_name,
    results_dir,
    out_dir=None,
    methods=("PCA", "SIMBA", "scVI", "SCENE"),
    seeds=range(5),
    error_bar="sem",
):
    results_dir = Path(results_dir)

    if out_dir is None:
        out_dir = results_dir / "benchmark_summary"
    else:
        out_dir = Path(out_dir)

    out_dir.mkdir(parents=True, exist_ok=True)

    display_names = {
        "PCA": "Harmony",
        "SIMBA": "SIMBA",
        "scVI": "scVI",
        "SCENE": "scLDM",
    }

    metric_candidates = [
        "Silhouette label",
        "KMeans NMI",
        "KMeans ARI",
        "Silhouette batch",
        "Graph connectivity",
    ]

    rows = []

    for method in methods:
        for seed in seeds:
            run_name = f"{method}_seed{seed}"
            csv_path = results_dir / run_name / "benchmark" / "benchmark_results.csv"

            if not csv_path.exists():
                raise FileNotFoundError(csv_path)

            df = pd.read_csv(csv_path)

            if len(df) != 1:
                raise ValueError(f"Expected one row in {csv_path}, got {len(df)}")

            row = df.iloc[0].to_dict()
            row["dataset"] = dataset_name
            row["method"] = method
            row["display_name"] = display_names.get(method, method)
            row["seed"] = seed
            row["run_name"] = run_name
            rows.append(row)

    if not rows:
        raise RuntimeError(f"No benchmark_results.csv files found in {results_dir}")

    all_results = pd.DataFrame(rows)

    metric_cols = [m for m in metric_candidates if m in all_results.columns]

    for col in metric_cols:
        all_results[col] = pd.to_numeric(all_results[col], errors="coerce")

    summary_rows = []

    for method in methods:
        g = all_results[all_results["method"] == method]

        row = {
            "dataset": dataset_name,
            "method": method,
            "display_name": display_names.get(method, method),
            "n_seeds": g["seed"].nunique(),
        }

        for metric in metric_cols:
            row[f"{metric}_mean"] = g[metric].mean()
            row[f"{metric}_std"] = get_error(g[metric], "std")
            row[f"{metric}_sem"] = get_error(g[metric], "sem")
            row[f"{metric}_ci95"] = get_error(g[metric], "ci95")
            row[f"{metric}_n"] = g[metric].notna().sum()

        summary_rows.append(row)

    summary = pd.DataFrame(summary_rows)

    dataset_safe = dataset_name.replace(" ", "_").lower()

    all_results_path = out_dir / f"benchmark_all_seeds_{dataset_safe}.csv"
    summary_path = out_dir / f"benchmark_summary_{dataset_safe}_mean_{error_bar}.csv"

    all_results.to_csv(all_results_path, index=False)
    summary.to_csv(summary_path, index=False)

    print(f"Saved seed-level results to: {all_results_path}")
    print(f"Saved summary to: {summary_path}")

    return all_results, summary, summary_path

In [ ]:
all_cortex, summary_cortex, cortex_summary_path = summarize_benchmark_dataset(
    dataset_name="CORTEX",
    results_dir=PROJECT_ROOT / "results" / "cortex",
    error_bar="std",
)

all_pbmc, summary_pbmc, pbmc_summary_path = summarize_benchmark_dataset(
    dataset_name="PBMC CITEseq",
    results_dir=PROJECT_ROOT / "results" / "pbmc_cite_seq",
    error_bar="std",
)

all_hca, summary_hca, hca_summary_path = summarize_benchmark_dataset(
    dataset_name="HCA Nuclei",
    results_dir=PROJECT_ROOT / "results" / "hca_nuclei",
    error_bar="std",
)

all_sim1, summary_sim1, sim1_summary_path = summarize_benchmark_dataset(
    dataset_name="SIM1",
    results_dir=PROJECT_ROOT / "results" / "sim_1",
    error_bar="std",
)

all_sim2, summary_sim2, sim2_summary_path = summarize_benchmark_dataset(
    dataset_name="SIM2",
    results_dir=PROJECT_ROOT / "results" / "sim_2",
    error_bar="std",
)


all_neurips, summary_neurips, neurips_summary_path = summarize_benchmark_dataset(
    dataset_name="Neurips",
    results_dir=PROJECT_ROOT / "results" / "neurips_cite",
    error_bar="std",
)

## Plotting

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import FormatStrFormatter


def set_nature_plot_style():
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
        "axes.linewidth": 0.5,
        "axes.labelsize": 5.6,
        "axes.titlesize": 5.8,
        "xtick.labelsize": 5.0,
        "ytick.labelsize": 5.0,
        "legend.fontsize": 5.0,
        "xtick.major.width": 0.5,
        "ytick.major.width": 0.5,
        "xtick.major.size": 2.0,
        "ytick.major.size": 2.0,
        "figure.dpi": 300,
        "savefig.dpi": 300,
    })


def style_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.grid(axis="y", color="#D9D9D9", linewidth=0.35)
    ax.grid(axis="x", visible=False)

    ax.tick_params(axis="both", which="both", pad=1.0)
    ax.set_axisbelow(True)

    ax.set_ylim(0.0, 1.01)
    ax.set_yticks(np.arange(0.0, 1.01, 0.2))
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))


def plot_benchmark_summaries(
    summary_csvs,
    out_dir,
    error_bar="sem",
    methods=("PCA", "SIMBA", "scVI", "SCENE"),
):
    set_nature_plot_style()

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    summary_dfs = []

    for path in summary_csvs:
        path = Path(path)

        if not path.exists():
            raise FileNotFoundError(path)

        summary_dfs.append(pd.read_csv(path))

    summary = pd.concat(summary_dfs, ignore_index=True)

    metrics = [
        ("Silhouette label", "Sil. Labels", "sil_labels_grouped_bar.svg"),
        ("KMeans NMI", "NMI", "nmi_grouped_bar.svg"),
        ("KMeans ARI", "ARI", "ari_grouped_bar.svg"),
        ("Silhouette batch", "Sil. Batch", "sil_batch_grouped_bar.svg"),
        ("Graph connectivity", "Graph Conn.", "graph_conn_grouped_bar.svg"),
    ]

    colors = {
        "PCA": "#7A7A7A",
        "SIMBA": "#0072B2",
        "scVI": "#D55E00",
        "SCENE": "#009E73",
    }

    display_names = {
        "Harmony": "Harmony",
        "SIMBA": "SIMBA",
        "scVI": "scVI",
        "SCENE": "SCENE",
    }

    datasets = summary["dataset"].drop_duplicates().tolist()
    group_centers = np.arange(len(datasets))

    bar_width = 0.14
    offsets = np.linspace(
        -bar_width * (len(methods) - 1) / 2,
        bar_width * (len(methods) - 1) / 2,
        len(methods),
    )

    def plot_metric(metric_col, ylabel, filename):
        mean_col = f"{metric_col}_mean"
        err_col = f"{metric_col}_{error_bar}"

        if mean_col not in summary.columns:
            print(f"Skipping missing metric: {metric_col}")
            return

        if err_col not in summary.columns:
            raise ValueError(f"Missing error column: {err_col}")

        fig, ax = plt.subplots(figsize=(1.26, 0.98), constrained_layout=True)

        for i, method in enumerate(methods):
            vals = []
            errs = []

            for dataset_name in datasets:
                sub = summary[
                    (summary["dataset"] == dataset_name)
                    & (summary["method"] == method)
                ]

                if sub.empty:
                    vals.append(np.nan)
                    errs.append(np.nan)
                    continue

                vals.append(sub[mean_col].iloc[0])
                errs.append(sub[err_col].iloc[0])

            ax.bar(
                group_centers + offsets[i],
                vals,
                width=bar_width,
                yerr=errs,
                capsize=0.9,
                error_kw={
                    "elinewidth": 0.30,
                    "capthick": 0.30,
                    "ecolor": "black",
                },
                color=colors[method],
                edgecolor="none",
                zorder=3,
            )

        ax.set_xticks(group_centers)
        ax.set_xticklabels(datasets, rotation=22, ha="right")
        ax.set_ylabel(ylabel, labelpad=1.0)

        ax.set_xlim(group_centers[0] - 0.42, group_centers[-1] + 0.42)

        style_axis(ax)

        outpath = out_dir / filename
        fig.savefig(outpath, bbox_inches="tight", transparent=True)
        fig.savefig(outpath.with_suffix(".pdf"), bbox_inches="tight", transparent=True)

        plt.close(fig)
        print(f"Saved: {outpath}")

    def save_standalone_legend():
        handles = [
            Patch(
                facecolor=colors[m],
                edgecolor="none",
                label=display_names.get(m, m),
            )
            for m in methods
        ]

        fig = plt.figure(figsize=(1.9, 0.24))
        fig.legend(
            handles=handles,
            labels=[display_names.get(m, m) for m in methods],
            loc="center",
            ncol=len(methods),
            frameon=False,
            handlelength=0.9,
            columnspacing=0.7,
            handletextpad=0.3,
            borderpad=0.0,
        )

        legend_path = out_dir / "legend_methods.svg"
        fig.savefig(legend_path, bbox_inches="tight", transparent=True)
        fig.savefig(legend_path.with_suffix(".pdf"), bbox_inches="tight", transparent=True)

        plt.close(fig)
        print(f"Saved: {legend_path}")

    for metric_col, ylabel, filename in metrics:
        plot_metric(metric_col, ylabel, filename)

    save_standalone_legend()

    print(f"Saved plots to: {out_dir}")

    return summary

In [ ]:
all_paths = [ hca_summary_path, neurips_summary_path, sim1_summary_path, sim2_summary_path ]

plot_benchmark_summaries(all_paths, OUT_DIR, error_bar="std")   

In [ ]:
def make_benchmark_summaries(
    summary_csvs,
    out_dir,
):
    set_nature_plot_style()

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    summary_dfs = []

    for path in summary_csvs:
        path = Path(path)

        if not path.exists():
            raise FileNotFoundError(path)

        summary_dfs.append(pd.read_csv(path))

    summary = pd.concat(summary_dfs, ignore_index=True)
    
    combined_summary_path = out_dir / f"benchmark_summary_combined.csv"
    summary.to_csv(combined_summary_path, index=False)

In [ ]:
all_paths = [cortex_summary_path, pbmc_summary_path,hca_summary_path, neurips_summary_path, sim1_summary_path, sim2_summary_path]


make_benchmark_summaries(summary_csvs=all_paths, out_dir=OUT_DIR,)